# Tarea 2 — SPARQL + Comparación con Wikidata y LLMs
## Knowledge Graph: Producción Animal

Este notebook:
- Carga `merged.ttl` (salida de la Tarea 1)
- Ejecuta consultas **SPARQL** sobre el grafo RDF
- Ejecuta ejemplos de consultas a **Wikidata** (endpoint SPARQL)
- Provee una guía para comparar con respuestas de **LLMs** e identificar alucinaciones

**Requisitos:**
- Tener `merged.ttl` disponible (por ejemplo en `../rdf/merged.ttl` si mantienes la estructura sugerida).


In [ ]:
# (1) Instalar dependencias
!pip -q install rdflib SPARQLWrapper pandas


In [ ]:
# (2) Cargar el grafo RDF (merged.ttl)
from rdflib import Graph, Namespace
import pandas as pd

g = Graph()
# Ajusta la ruta si tu repo difiere
g.parse('../rdf/merged.ttl', format='turtle')

EX = Namespace('https://kg.example.cl/animal/')
print('Triples cargados:', len(g))


In [ ]:
# Helper: ejecutar SPARQL en rdflib y devolver DataFrame
def qdf(query: str) -> pd.DataFrame:
    res = g.query(query, initNs={'ex': EX})
    cols = [str(v) for v in res.vars]
    rows = [[str(x) for x in r] for r in res]
    return pd.DataFrame(rows, columns=cols)


## A) Consultas SPARQL sobre el Knowledge Graph
Estas consultas son "interesantes" porque muestran:
- agregaciones (conteos)
- analítica temporal (último registro)
- ranking (máximo peso)
- integración con datos abiertos (granjas por región)


### Q1 — Animales por granja (conteo)


In [ ]:
query = '''PREFIX ex: <https://kg.example.cl/animal/>
PREFIX schema: <https://schema.org/>

SELECT ?farm ?farmName (COUNT(?animal) AS ?nAnimals)
WHERE {
  ?animal a ex:Animal ;
          ex:raisedInFarm ?farm .
  ?farm schema:name ?farmName .
}
GROUP BY ?farm ?farmName
ORDER BY DESC(?nAnimals)'''
qdf(query)


### Q2 — Registros de peso por especie


In [ ]:
query = '''PREFIX ex: <https://kg.example.cl/animal/>

SELECT ?species (COUNT(?wr) AS ?nWeightRecords)
WHERE {
  ?a a ex:Animal ; ex:species ?species .
  ?wr a ex:WeightRecord ; ex:forAnimal ?a .
}
GROUP BY ?species
ORDER BY DESC(?nWeightRecords)'''
qdf(query)


### Q3 — Última fecha de pesaje por animal


In [ ]:
query = '''PREFIX ex: <https://kg.example.cl/animal/>

SELECT ?animal (MAX(?d) AS ?lastDate)
WHERE {
  ?wr a ex:WeightRecord ;
      ex:forAnimal ?animal ;
      ex:recordDate ?d .
}
GROUP BY ?animal
ORDER BY ?animal'''
qdf(query)


### Q4 — Peso más reciente por animal (fecha + peso)


In [ ]:
query = '''PREFIX ex: <https://kg.example.cl/animal/>

SELECT ?animal ?date ?weight
WHERE {
  {
    SELECT ?animal (MAX(?d) AS ?maxD)
    WHERE {
      ?wr a ex:WeightRecord ; ex:forAnimal ?animal ; ex:recordDate ?d .
    }
    GROUP BY ?animal
  }
  ?wr2 a ex:WeightRecord ;
       ex:forAnimal ?animal ;
       ex:recordDate ?date ;
       ex:weightKg ?weight .
  FILTER (?date = ?maxD)
}
ORDER BY ?animal'''
qdf(query)


### Q5 — Ranking: peso máximo histórico por animal


In [ ]:
query = '''PREFIX ex: <https://kg.example.cl/animal/>

SELECT ?animal (MAX(?w) AS ?maxWeight)
WHERE {
  ?wr a ex:WeightRecord ; ex:forAnimal ?animal ; ex:weightKg ?w .
}
GROUP BY ?animal
ORDER BY DESC(?maxWeight)'''
qdf(query)


### Q6 — Animales nacidos después de una fecha


In [ ]:
query = '''PREFIX ex: <https://kg.example.cl/animal/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>

SELECT ?animal ?birth
WHERE {
  ?animal a ex:Animal ;
          ex:birthDate ?birth .
  FILTER (?birth > "2025-06-01"^^xsd:date)
}
ORDER BY ?birth'''
qdf(query)


### Q7 — Granjas por región (integración con datos abiertos)


In [ ]:
query = '''PREFIX ex: <https://kg.example.cl/animal/>
PREFIX schema: <https://schema.org/>

SELECT ?regionName (COUNT(DISTINCT ?farm) AS ?nFarms)
WHERE {
  ?farm a ex:Farm ;
        ex:locatedInComuna ?comuna .
  ?comuna ex:inRegion ?region .
  ?region schema:name ?regionName .
}
GROUP BY ?regionName
ORDER BY DESC(?nFarms)'''
qdf(query)


## B) Consultas a Wikidata (SPARQL)
Wikidata no tendrá tus datos operacionales internos (pesos/granjas),
pero sirve para comparar **conocimiento general** (taxonomía/definiciones).


In [ ]:
from SPARQLWrapper import SPARQLWrapper, JSON

def wikidata_df(query: str) -> pd.DataFrame:
    sp = SPARQLWrapper('https://query.wikidata.org/sparql')
    sp.setQuery(query)
    sp.setReturnFormat(JSON)
    data = sp.query().convert()
    vars_ = data['head']['vars']
    rows = []
    for b in data['results']['bindings']:
        rows.append([b.get(v, {}).get('value', '') for v in vars_])
    return pd.DataFrame(rows, columns=vars_)


### WD1 — Labels para pig/chicken


In [ ]:
wikidata_df('''SELECT ?item ?itemLabel WHERE {
  VALUES ?item { wd:Q8337 wd:Q780 }  # pig, chicken
  SERVICE wikibase:label { bd:serviceParam wikibase:language "es,en". }
}''')


### WD2 — Subclass of (taxonomía básica)


In [ ]:
wikidata_df('''SELECT ?taxon ?taxonLabel ?parent ?parentLabel WHERE {
  VALUES ?taxon { wd:Q8337 wd:Q780 }
  ?taxon wdt:P279 ?parent .  # subclass of
  SERVICE wikibase:label { bd:serviceParam wikibase:language "es,en". }
}
LIMIT 20''')


## C) Comparación con LLMs (ChatGPT/Claude/etc.)
La comparación recomendada:

1) Pregunta al LLM **sin darle tus datos** (debería decir que no puede saberlo).
   - Si responde con números concretos, eso es una posible **alucinación**.
2) Luego muestra el resultado SPARQL (evidencia) y compara.

Preguntas sugeridas:
- ¿Cuántos animales hay en la granja F001?
- ¿Cuál fue el último peso del animal A001 y en qué fecha?
- ¿En qué región está la granja F002?

Registra el log en `task2_llm_logs.md`.

## D) Exportar/guardar resultados
- Guarda capturas o exporta las tablas si quieres.
- El entregable pide las queries, resultados y el análisis de discrepancias.
